In [1]:

# User Question
#       ↓
#  ┌────┴─────┐
#  ↓          ↓
# FAISS      BM25
# Top-10     Top-10
#  ↓          ↓
#  └────┬─────┘
#       ↓
# Combine candidates
#       ↓
# Cross-Encoder Reranker
#       ↓
# Top-5
#       ↓
# Evaluation


<!-- User Question
      ↓
 ┌────┴─────┐
 ↓          ↓
FAISS      BM25
Top-10     Top-10
 ↓          ↓
 └────┬─────┘
      ↓
Combine candidates
      ↓
Cross-Encoder Reranker
      ↓
Top-5
      ↓
Evaluation -->


It uses:

- FAISS: all-MiniLM-L6-v2
- BM25: rank_bm25
- Reranker: cross-encoder/ms-marco-MiniLM-L-6-v2
- The same test_prompts.json
- Your existing chunks and Iteration 2 embedding approach


Iteration 3

- Dense retrieval with FAISS: Top-10
- Keyword retrieval with BM25: Top-10
- Combine candidates
- Cross-encoder reranking
- Final Top-5
- Evaluate Any-hit and All-hit
- Compare Iteration 2 baseline against Iteration 3


In [2]:
# Install dependencies
!pip -q install sentence-transformers faiss-cpu rank-bm25 pandas numpy



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import json
import re
import numpy as np
import pandas as pd
import faiss

from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
from IPython.display import display

print("Libraries loaded successfully.")


c:\Users\KRUSHIL\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Libraries loaded successfully.


In [4]:
# File paths
chunks_path = "./02_rmit_assessment_support_chunks.csv"
test_path = "./test_prompts.json"

# Load chunks
chunks_df = pd.read_csv(chunks_path)

# Load test prompts
with open(test_path, "r", encoding="utf-8") as f:
    test_data = json.load(f)

test_df = pd.DataFrame(test_data["tests"])

print("Number of chunks:", len(chunks_df))
print("Number of test prompts:", len(test_df))

display(test_df)


Number of chunks: 71
Number of test prompts: 12


,prompt,chunks
0,I've got a timed quiz that I won't be able to ...,[EXT-01_1]
1,"I've recently lost my job, is this a valid rea...",[SC-01_7]
2,So my hard drive with my assignment stopped wo...,[SC-01_16]
3,I've just booked my first appointment with ELS...,[EAA-02_29]
4,I emailed my course coordinator three days ago...,[EAA-02_38]
5,Having a hard time reading through journal art...,[EAA-02_46]
6,My mate got flagged for plagiarism because he ...,[POL-01_70]
7,Who actually sets RMIT's official academic cal...,[POL-01_67]
8,"I missed a test because I was unwell, How long...","[SC-01_8, EAA-02_43]"
9,My nan just passed away but I've got a group a...,"[POL-01_62, EXT-01_1]"


Dense retrieval — FAISS


In [5]:
# Create the same embedding text used in Iteration 2
chunks_df["embedding_text"] = (
    chunks_df["category"].fillna("") + ". " +
    chunks_df["section"].fillna("") + ". " +
    chunks_df["content"].fillna("")
)

# Load the same embedding model
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# Create document embeddings
embeddings = embedding_model.encode(
    chunks_df["embedding_text"].tolist(),
    show_progress_bar=True,
    convert_to_numpy=True
).astype("float32")

faiss.normalize_L2(embeddings)

# Build FAISS index
dimension = embeddings.shape[1]
faiss_index = faiss.IndexFlatIP(dimension)
faiss_index.add(embeddings)

print("FAISS index created.")
print("Vectors:", faiss_index.ntotal)
print("Dimension:", dimension)


Batches: 100%|██████████| 3/3 [00:00<00:00,  3.26it/s]

FAISS index created.
Vectors: 71
Dimension: 384


### BM25 keyword retrieval

BM25 provides a second retrieval signal based on exact/lexical terms.
This is useful for questions containing specific terms such as:

- hard drive
- plagiarism
- dyslexia
- extension
- special consideration
  The BM25 search returns the top 10 candidates.


In [6]:
def tokenize(text):
    return re.findall(r"\b\w+\b", str(text).lower())

bm25_corpus = [
    tokenize(text)
    for text in chunks_df["embedding_text"].tolist()
]

bm25 = BM25Okapi(bm25_corpus)

print("BM25 index created.")
print("Documents:", len(bm25_corpus))


BM25 index created.
Documents: 71


### Cross-encoder reranker

The reranker reads the question and candidate chunk together, rather than comparing two independent embeddings.
We first retrieve candidates with FAISS and BM25, then let the cross-encoder decide which candidates are most relevant.


In [7]:
reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

print("Cross-encoder reranker loaded.")


c:\Users\KRUSHIL\AppData\Local\Programs\Python\Python314\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\KRUSHIL\.cache\huggingface\hub\models--cross-encoder--ms-marco-MiniLM-L-6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\KRUSHIL\AppData\Local\Programs\Python\Python314\Lib\site-package

Cross-encoder reranker loaded.


In [ ]:
def dense_candidates(question, k=10):
    query_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True
    ).astype("float32")

    faiss.normalize_L2(query_embedding)

    scores, indices = faiss_index.search(
        query_embedding,
        k
    )

    return [
        {
            "index": int(idx),
            "dense_score": float(score)
        }
        for idx, score in zip(indices[0], scores[0])
    ]


def bm25_candidates(question, k=10):
    query_tokens = tokenize(question)

    scores = bm25.get_scores(query_tokens)

    # Highest BM25 scores first
    top_indices = np.argsort(scores)[::-1][:k]

    return [
        {
            "index": int(idx),
            "bm25_score": float(scores[idx])
        }
        for idx in top_indices
    ]


def hybrid_rerank(question, dense_k=10, bm25_k=10, final_k=5):
    dense_results = dense_candidates(question, dense_k)
    bm25_results = bm25_candidates(question, bm25_k)

    # Union of candidates from both retrieval methods
    candidate_indices = set()

    for result in dense_results:
        candidate_indices.add(result["index"])

    for result in bm25_results:
        candidate_indices.add(result["index"])

    candidate_indices = list(candidate_indices)

    # Cross-encoder pairs
    pairs = [
        (
            question,
            chunks_df.iloc[idx]["embedding_text"]
        )
        for idx in candidate_indices
    ]

    rerank_scores = reranker.predict(pairs)

    ranked = sorted(
        zip(candidate_indices, rerank_scores),
        key=lambda x: float(x[1]),
        reverse=True
    )

    final_results = []

    for idx, rerank_score in ranked[:final_k]:
        row = chunks_df.iloc[idx]

        dense_score = next(
            (
                x["dense_score"]
                for x in dense_results
                if x["index"] == idx
            ),
            None
        )

        bm25_score = next(
            (
                x["bm25_score"]
                for x in bm25_results
                if x["index"] == idx
            ),
            None
        )

        final_results.append({
            "index": idx,
            "chunk_id": row["chunk_id"],
            "source_id": row["source_id"],
            "category": row["category"],
            "title": row["title"],
            "section": row["section"],
            "content": row["content"],
            "dense_score": dense_score,
            "bm25_score": bm25_score,
            "rerank_score": float(rerank_score)
        })

    return pd.DataFrame(final_results)




In [20]:
def diagnose_retrieval(question, expected_chunks, dense_k=20, bm25_k=20):

    dense_results = dense_candidates(question, dense_k)
    bm25_results = bm25_candidates(question, bm25_k)

    dense_ids = [
        chunks_df.iloc[x["index"]]["chunk_id"]
        for x in dense_results
    ]

    bm25_ids = [
        chunks_df.iloc[x["index"]]["chunk_id"]
        for x in bm25_results
    ]

    candidate_ids = list(set(dense_ids + bm25_ids))

    rows = []

    for expected in expected_chunks:

        rows.append({
            "expected_chunk": expected,
            "in_faiss_top20": expected in dense_ids,
            "in_bm25_top20": expected in bm25_ids,
            "in_candidate_pool": expected in candidate_ids
        })

    return pd.DataFrame(rows)

In [11]:
#Testing one question
question = "I've recently lost my job, is this a valid reason to apply for special consideration?"

sample_results = hybrid_rerank(
    question,
    dense_k=10,
    bm25_k=10,
    final_k=5
)

display(
    sample_results[
        [
            "chunk_id",
            "source_id",
            "section",
            "dense_score",
            "bm25_score",
            "rerank_score",
            "content"
        ]
    ]
)

,chunk_id,source_id,section,dense_score,bm25_score,rerank_score,content
0,SC-01_7,SC-01,Eligibility,0.463712,NaN,3.845328,If unexpected circumstances outside your contr...
1,SC-01_13,SC-01,False documents and misleading information,0.449625,10.065597,3.362396,be significantly impacted by grief for some ti...
2,SC-01_4,SC-01,Special consideration,0.493858,NaN,3.345960,If unexpected circumstances beyond your contro...
3,SC-01_6,SC-01,What is special consideration?,0.410347,9.453736,3.312253,If unexpected circumstances outside your contr...
4,SC-01_9,SC-01,How to apply,0.371078,NaN,2.245075,of your application. If your application is in...


### Evaluate Iteration 3

For each test question we record:

- Expected chunk IDs
- Top-5 retrieved chunk IDs
- Reranker scores
- Whether any expected chunk appears in Top-1/3/5
- Whether all expected chunks appear in Top-1/3/5
  This allows direct comparison with the Iteration 2 baseline.


In [12]:
def any_expected_retrieved(retrieved, expected):
    return any(chunk in retrieved for chunk in expected)


def all_expected_retrieved(retrieved, expected):
    return all(chunk in retrieved for chunk in expected)


def evaluate_iteration3(test_df):
    results = []

    for _, test in test_df.iterrows():

        question = test["prompt"]
        expected_chunks = test["chunks"]

        retrieved_df = hybrid_rerank(
            question,
            dense_k=10,
            bm25_k=10,
            final_k=5
        )

        retrieved_chunks = retrieved_df["chunk_id"].tolist()

        top1 = retrieved_chunks[:1]
        top3 = retrieved_chunks[:3]
        top5 = retrieved_chunks[:5]

        expected_ranks = [
            retrieved_chunks.index(chunk) + 1
            for chunk in expected_chunks
            if chunk in retrieved_chunks
        ]

        results.append({
            "prompt": question,
            "expected_chunks": expected_chunks,

            "top1_chunk": top1[0] if top1 else None,
            "top3_chunks": top3,
            "top5_chunks": top5,

            "expected_rank_in_top5":
                min(expected_ranks) if expected_ranks else None,

            # Any expected chunk
            "top1_hit":
                any_expected_retrieved(top1, expected_chunks),
            "top3_hit":
                any_expected_retrieved(top3, expected_chunks),
            "top5_hit":
                any_expected_retrieved(top5, expected_chunks),

            # All expected chunks
            "top1_all_hit":
                all_expected_retrieved(top1, expected_chunks),
            "top3_all_hit":
                all_expected_retrieved(top3, expected_chunks),
            "top5_all_hit":
                all_expected_retrieved(top5, expected_chunks)
        })

    return pd.DataFrame(results)



In [13]:
# Run Iteration 3 evaluation
iteration3_df = evaluate_iteration3(test_df)

display(iteration3_df)


,prompt,expected_chunks,top1_chunk,top3_chunks,top5_chunks,expected_rank_in_top5,top1_hit,top3_hit,top5_hit,top1_all_hit,top3_all_hit,top5_all_hit
0,I've got a timed quiz that I won't be able to ...,[EXT-01_1],EXT-01_1,"[EXT-01_1, EXT-01_0, EAA-02_38]","[EXT-01_1, EXT-01_0, EAA-02_38, EAA-02_43, EXT...",1.0,True,True,True,True,True,True
1,"I've recently lost my job, is this a valid rea...",[SC-01_7],SC-01_7,"[SC-01_7, SC-01_13, SC-01_4]","[SC-01_7, SC-01_13, SC-01_4, SC-01_6, SC-01_9]",1.0,True,True,True,True,True,True
2,So my hard drive with my assignment stopped wo...,[SC-01_16],EAA-02_40,"[EAA-02_40, SC-01_16, SC-01_6]","[EAA-02_40, SC-01_16, SC-01_6, EXT-01_0, EXT-0...",2.0,False,True,True,False,True,True
3,I've just booked my first appointment with ELS...,[EAA-02_29],EAA-02_29,"[EAA-02_29, EAA-01_18, EAA-02_44]","[EAA-02_29, EAA-01_18, EAA-02_44, EAA-02_21, E...",1.0,True,True,True,True,True,True
4,I emailed my course coordinator three days ago...,[EAA-02_38],EAA-02_34,"[EAA-02_34, EAA-02_38, EXT-01_2]","[EAA-02_34, EAA-02_38, EXT-01_2, EXT-01_0, EAA...",2.0,False,True,True,False,True,True
5,Having a hard time reading through journal art...,[EAA-02_46],EAA-02_46,"[EAA-02_46, EAA-02_51, SC-01_16]","[EAA-02_46, EAA-02_51, SC-01_16, SC-01_8, EAA-...",1.0,True,True,True,True,True,True
6,My mate got flagged for plagiarism because he ...,[POL-01_70],POL-01_70,"[POL-01_70, EXT-01_3, POL-01_56]","[POL-01_70, EXT-01_3, POL-01_56, SC-01_8, EAA-...",1.0,True,True,True,True,True,True
7,Who actually sets RMIT's official academic cal...,[POL-01_67],POL-01_67,"[POL-01_67, SC-01_8, EAA-02_27]","[POL-01_67, SC-01_8, EAA-02_27, POL-01_70, EAA...",1.0,True,True,True,True,True,True
8,"I missed a test because I was unwell, How long...","[SC-01_8, EAA-02_43]",EAA-02_43,"[EAA-02_43, SC-01_8, POL-01_63]","[EAA-02_43, SC-01_8, POL-01_63, EXT-01_0, SC-0...",1.0,True,True,True,False,True,True
9,My nan just passed away but I've got a group a...,"[POL-01_62, EXT-01_1]",EAA-02_41,"[EAA-02_41, EAA-02_34, EXT-01_0]","[EAA-02_41, EAA-02_34, EXT-01_0, EAA-02_36, EA...",NaN,False,False,False,False,False,False


In [14]:
print("ITERATION 3 — HYBRID + RERANKING")
print("----------------------------------")

print("\nAny expected chunk retrieved:")
print(f"Top-1: {iteration3_df['top1_hit'].mean():.1%}")
print(f"Top-3: {iteration3_df['top3_hit'].mean():.1%}")
print(f"Top-5: {iteration3_df['top5_hit'].mean():.1%}")

print("\nAll expected chunks retrieved:")
print(f"Top-1: {iteration3_df['top1_all_hit'].mean():.1%}")
print(f"Top-3: {iteration3_df['top3_all_hit'].mean():.1%}")
print(f"Top-5: {iteration3_df['top5_all_hit'].mean():.1%}")


ITERATION 3 — HYBRID + RERANKING
----------------------------------

Any expected chunk retrieved:
Top-1: 58.3%
Top-3: 75.0%
Top-5: 83.3%

All expected chunks retrieved:
Top-1: 50.0%
Top-3: 75.0%
Top-5: 75.0%


In [15]:
comparison_df = pd.DataFrame({
    "Metric": [
        "Any expected — Top-1",
        "Any expected — Top-3",
        "Any expected — Top-5",
        "All expected — Top-1",
        "All expected — Top-3",
        "All expected — Top-5"
    ],
    "Iteration 2": [
        0.500,
        0.750,
        0.917,
        0.417,
        0.583,
        0.750
    ],
    "Iteration 3": [
        iteration3_df["top1_hit"].mean(),
        iteration3_df["top3_hit"].mean(),
        iteration3_df["top5_hit"].mean(),
        iteration3_df["top1_all_hit"].mean(),
        iteration3_df["top3_all_hit"].mean(),
        iteration3_df["top5_all_hit"].mean()
    ]
})

comparison_df["Change"] = (
    comparison_df["Iteration 3"] -
    comparison_df["Iteration 2"]
)

display(comparison_df.style.format({
    "Iteration 2": "{:.1%}",
    "Iteration 3": "{:.1%}",
    "Change": "{:+.1%}"
}))


,Metric,Iteration 2,Iteration 3,Change
0,Any expected — Top-1,50.0%,58.3%,+8.3%
1,Any expected — Top-3,75.0%,75.0%,+0.0%
2,Any expected — Top-5,91.7%,83.3%,-8.4%
3,All expected — Top-1,41.7%,50.0%,+8.3%
4,All expected — Top-3,58.3%,75.0%,+16.7%
5,All expected — Top-5,75.0%,75.0%,+0.0%


In [16]:
# Identify remaining Top-1 failures
top1_failures_3 = iteration3_df[
    iteration3_df["top1_hit"] == False
][
    [
        "prompt",
        "expected_chunks",
        "top1_chunk",
        "top3_chunks",
        "top5_chunks",
        "expected_rank_in_top5"
    ]
]

print("Iteration 3 Top-1 failures:", len(top1_failures_3))
display(top1_failures_3)


Iteration 3 Top-1 failures: 5


,prompt,expected_chunks,top1_chunk,top3_chunks,top5_chunks,expected_rank_in_top5
2,So my hard drive with my assignment stopped wo...,[SC-01_16],EAA-02_40,"[EAA-02_40, SC-01_16, SC-01_6]","[EAA-02_40, SC-01_16, SC-01_6, EXT-01_0, EXT-0...",2.0
4,I emailed my course coordinator three days ago...,[EAA-02_38],EAA-02_34,"[EAA-02_34, EAA-02_38, EXT-01_2]","[EAA-02_34, EAA-02_38, EXT-01_2, EXT-01_0, EAA...",2.0
9,My nan just passed away but I've got a group a...,"[POL-01_62, EXT-01_1]",EAA-02_41,"[EAA-02_41, EAA-02_34, EXT-01_0]","[EAA-02_41, EAA-02_34, EXT-01_0, EAA-02_36, EA...",NaN
10,I think my tutor marked me down assuming I pla...,"[POL-01_58, POL-01_59]",SC-01_16,"[SC-01_16, POL-01_70, SC-01_11]","[SC-01_16, POL-01_70, SC-01_11, EAA-02_38, EAA...",NaN
11,"The extensions adjustment is on my ELP, but th...","[EXT-01_2, EAA-02_39]",EAA-02_34,"[EAA-02_34, EAA-02_36, EAA-02_41]","[EAA-02_34, EAA-02_36, EAA-02_41, EAA-02_37, E...",5.0


In [17]:
# Save detailed Iteration 3 results
iteration3_output = "./06_iteration3_hybrid_reranking_results.csv"
iteration3_df.to_csv(
    iteration3_output,
    index=False,
    encoding="utf-8-sig"
)

# Save comparison
comparison_output = "./06_iteration2_vs_iteration3_comparison.csv"
comparison_df.to_csv(
    comparison_output,
    index=False,
    encoding="utf-8-sig"
)

print("Saved:")
print(iteration3_output)
print(comparison_output)


Saved:
./06_iteration3_hybrid_reranking_results.csv
./06_iteration2_vs_iteration3_comparison.csv


In [18]:
top5_failures_3 = iteration3_df[
    iteration3_df["top5_hit"] == False
][[
    "prompt",
    "expected_chunks",
    "top1_chunk",
    "top3_chunks",
    "top5_chunks",
    "expected_rank_in_top5"
]]

print("Iteration 3 Top-5 failures:", len(top5_failures_3))
display(top5_failures_3)

Iteration 3 Top-5 failures: 2


,prompt,expected_chunks,top1_chunk,top3_chunks,top5_chunks,expected_rank_in_top5
9,My nan just passed away but I've got a group a...,"[POL-01_62, EXT-01_1]",EAA-02_41,"[EAA-02_41, EAA-02_34, EXT-01_0]","[EAA-02_41, EAA-02_34, EXT-01_0, EAA-02_36, EA...",NaN
10,I think my tutor marked me down assuming I pla...,"[POL-01_58, POL-01_59]",SC-01_16,"[SC-01_16, POL-01_70, SC-01_11]","[SC-01_16, POL-01_70, SC-01_11, EAA-02_38, EAA...",NaN


In [19]:
all_top5_failures_3 = iteration3_df[
    iteration3_df["top5_all_hit"] == False
][[
    "prompt",
    "expected_chunks",
    "top1_chunk",
    "top3_chunks",
    "top5_chunks",
    "expected_rank_in_top5"
]]

print("Iteration 3 All Top-5 failures:", len(all_top5_failures_3))
display(all_top5_failures_3)

Iteration 3 All Top-5 failures: 3


,prompt,expected_chunks,top1_chunk,top3_chunks,top5_chunks,expected_rank_in_top5
9,My nan just passed away but I've got a group a...,"[POL-01_62, EXT-01_1]",EAA-02_41,"[EAA-02_41, EAA-02_34, EXT-01_0]","[EAA-02_41, EAA-02_34, EXT-01_0, EAA-02_36, EA...",NaN
10,I think my tutor marked me down assuming I pla...,"[POL-01_58, POL-01_59]",SC-01_16,"[SC-01_16, POL-01_70, SC-01_11]","[SC-01_16, POL-01_70, SC-01_11, EAA-02_38, EAA...",NaN
11,"The extensions adjustment is on my ELP, but th...","[EXT-01_2, EAA-02_39]",EAA-02_34,"[EAA-02_34, EAA-02_36, EAA-02_41]","[EAA-02_34, EAA-02_36, EAA-02_41, EAA-02_37, E...",5.0


In [21]:
q1 = "My nan just passed away but I've got a group assignment due friday, can I even apply for an extension on something like this? How much extra time could I get?"

display(
    diagnose_retrieval(
        q1,
        ["POL-01_62", "EXT-01_1"]
    )
)

,expected_chunk,in_faiss_top20,in_bm25_top20,in_candidate_pool
0,POL-01_62,True,True,True
1,EXT-01_1,True,True,True


In [22]:
q2 = "I think my tutor marked me down assuming I plagiarised, very upset about this because I definitely didn't, how should I go about correcting this issue?"

display(
    diagnose_retrieval(
        q2,
        ["POL-01_58", "POL-01_59"]
    )
)

,expected_chunk,in_faiss_top20,in_bm25_top20,in_candidate_pool
0,POL-01_58,True,False,True
1,POL-01_59,True,False,True


In [23]:
q3 = "The extensions adjustment is on my ELP, but the assignment brief also says to fill out the standard extension form, do I need to do both, or is my learning plan enough on its own"

display(
    diagnose_retrieval(
        q3,
        ["EXT-01_2", "EAA-02_39"]
    )
)

,expected_chunk,in_faiss_top20,in_bm25_top20,in_candidate_pool
0,EXT-01_2,True,True,True
1,EAA-02_39,True,True,True
